# Dashboard Visualization Notebook
This notebook loads the datasets, summarizes the label distributions, and creates visualizations for a news credibility dashboard.

In [ ]:
import pandas as pd
import plotly.express as px
from sklearn.feature_extraction.text import CountVectorizer

fake = pd.read_csv('../data/Fake.csv')
real = pd.read_csv('../data/True.csv')
clickbait = pd.read_csv('../data/clickbait.csv')

fake['label'] = 'fake'
real['label'] = 'real'
clickbait['clickbait_label'] = clickbait['label'].map({1: 'clickbait', 0: 'not clickbait'})

display(fake.head(3))
display(real.head(3))
display(clickbait.head(3))

## Dataset Distribution
Visualize how the fake/real dataset and the clickbait dataset are balanced.

In [ ]:
news_counts = pd.concat([fake, real])[['label']].assign(dataset='news')
news_counts = news_counts['label'].value_counts().reset_index()
news_counts.columns = ['label', 'count']
fig1 = px.bar(news_counts, x='label', y='count', color='label', title='Fake vs Real News Distribution', color_discrete_map={'fake':'red','real':'green'})
fig1.show()

click_counts = clickbait['clickbait_label'].value_counts().reset_index()
click_counts.columns = ['label', 'count']
fig2 = px.pie(click_counts, names='label', values='count', title='Clickbait vs Non-Clickbait Distribution')
fig2.show()

## Text Insights
Compare the most common terms in fake and real news so the dashboard can highlight suspicious language.

In [ ]:
def top_terms(df, field, label, n=15):
    vectorizer = CountVectorizer(stop_words='english', max_features=2000)
    X = vectorizer.fit_transform(df[field].fillna(''))
    counts = X.sum(axis=0).A1
    terms = vectorizer.get_feature_names_out()
    result = pd.DataFrame({'term': terms, 'count': counts})
    result['label'] = label
    return result.sort_values('count', ascending=False).head(n)

fake_terms = top_terms(fake, 'text', 'fake')
real_terms = top_terms(real, 'text', 'real')
top_terms_df = pd.concat([fake_terms, real_terms])

fig3 = px.bar(top_terms_df, x='term', y='count', color='label', barmode='group', title='Top Terms in Fake vs Real News')
fig3.update_layout(xaxis_tickangle=-45)
fig3.show()

## Suspicious Word Highlighting
This section can be adapted to highlight suspicious words inside a sentence in your dashboard app.